##### Import Libaries and Functions

In [0]:
# import boto3
import os

from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col
from pyspark.sql import DataFrame

import delta_table_creator, transform
from delta import DeltaTable  # type: ignore

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix

env = aws_utils.get_env_prefix()

##### Create widgets with default values. Change them if needed from UI
* run_date : '2024-02-28'
* unity : True | False

In case of Unity Catalog, the appropriate cluster should be attached

##### Get the values of widgets and initialize the environment variable
Ensure that widgets are assigned the wanted values before proceeding with the next step.

In [0]:
# Takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
#env = os.environ["S3PREFIX"] 
# Takes the values of widgets
run_date = dbutils.widgets.get("run_date") 
unity = dbutils.widgets.get("unity") 

table_name = 'DParty'

# """
# Takes the s3 parameter value from ssm

# ssm_client = boto3.client("ssm", region_name="eu-central-1") # Initialize the SSM client

# response = ssm_client.get_parameter(
#     Name='/cfn/a360/matching/conf_s3_path',
#     WithDecryption=False
# )

# s3_path = response['Parameter']['Value']
# """

##### Initialize the schema retrieved from Redis

In [0]:
schema = (
    "PartyDerivativeId string"
    ",SourceSystem string"
    ",PortfolioKey string"
    ",PortfolioName string"
    ",IngestionTimestamp timestamp"
    ",BatchId string"
    ",PartyKey string"
    ",PartyKeyQCCepal string"
    ",PartyKeyProfits string"
    ",PartyKeyArotron string"
    ",PartyKeyThesis string"
    ",PartyKeyQCAlpha string"
    ",PartyKeyLMS string"
    ",CDI string"
    ",PartyTypeCode string"
    ",PartyType string"
    ",PartyStatusCode string"
    ",PartyStatus string"
    ",PartyStatusDate date"
    ",L3869StatusCode string"
    ",L3869Status string"
    ",L3869CourtDate date"
    ",L3869InterbankingStatusCode int"
    ",L3869InterBankingStatus string"
    ",OCWFlag boolean"
    ",OCWStatus string"
    ",DeceasedFlag boolean"
    ",LPSCode int"
    ",LPS string"
    ",LPSDate date"
    ",Watchlist string"
    ",FirstNameGR string"
    ",FirstNameENG string"
    ",LastNameGR string"
    ",LastNameENG string"
    ",FirstFatherNameGR string"
    ",FirstFatherNameENG string"
    ",LastFatherNameGR string"
    ",LastFatherNameENG string"
    ",FirstMotherNameGR string"
    ",LastMotherNameGR string"
    ",LastMotherNameENG string"
    ",LegalNameGR string"
    ",LegalNameENG string"
    ",PrimaryCommercialGR string"
    ",PrimaryCommercialENG string"
    ",SalutationTypeCode string"
    ",SalutationType string"
    ",GenderCode string"
    ",Gender string"
    ",MaritalStatusCode string"
    ",MaritalStatus string"
    ",MaritalStatusENG string"
    ",CitizenshipCode string"
    ",Citizenship string"
    ",NationalityCode string"
    ",Nationality string"
    ",NationalityENG string"
    ",OccupationCode string"
    ",Occupation string"
    ",BirthDate date"
    ",BirthPlace string"
    ",CommunicationLanguageCode int"
    ",CommunicationLanguage string"
    ",TaxRegistrationNumber string"
    ",TaxResidenceCountry string"
    ",PassportNumber string"
    ",PassportIssuingAuthority string"
    ",PassportIssueDate date"
    ",PassportExpiredDate date"
    ",NationalNumber string"
    ",NationalNumberIssuingAuthority string"
    ",NationalNumberIssueDate date"
    ",NationalNumberExpiredDate date"
    ",MilitaryNumber string"
    ",MilitaryAuthority string"
    ",MilitaryIssueDate date"
    ",MilitaryExpiredDate date"
    ",SocialSecurityNumber string"
    ",NumberOfChildren string"
    ",ChildrenCertification string"
    ",ChildrenCertificationIssuingAuthority string"
    ",EnterpriseStartDate date"
    ",EnterpriseEndDate date"
    ",EconomicActivityCode int"
    ",EconomicActivity string"
    ",BranchActivityCode int"
    ",BranchActivity string"
    ",GeneralCommercialRegister string"
    ",NumberofEmployees string"
    ",TotalIncome string"
    ",BalanceSheetTotalAmount string"
    ",AnnualTurnoverAmount string"
    ",GroupCode string"
    ",GroupName string"
    ",GroupCodePortfolio int"
    ",PartyBankruptcyStage int"
    ",CovidFlag string"
    ",CovidCategoryCode int"
    ",CovidCategory string"
    ",AffectedCategoryCode int"
    ",AffectedCategory string"
    ",MonitoringUnitCode string"
    ",MonitoringUnit string"
    ",CreditRatingTypeCode int"
    ",CreditRatingType string"
    ",EducationLevelCode int"
    ",EducationLevel string"
    ",BalanceCategoryCode string"
    ",BalanceCategory string"
    ",ExtraSegmentCode string"
    ",ExtraSegment string"
    ",SegmentCode string"
    ",Segment string"
    ",SubSegmentCode string"
    ",SubSegment string"
    ",LegalFormCode int"
    ",LegalForm string"
    ",IsWholesaleFlag boolean"
    ",EmploymentStatusCode int"
    ",EmploymentStatus string"
    ",EmploymentStatusDate date"
    ",CreatedBy string"
    ",CreatedByDate timestamp"
    ",LastUpdatedBy string"
    ",LastUpdatedByDate timestamp"
    ",ResubmissionStatus string"
    ",N4605Flag int"
    ",ConnectedBorrowerFlag boolean"
    ",StaffFlag string"
    ",L3869BorrowerHasOver50KBalance boolean"
    ",L3869Flag boolean"
    ",PartyRegion string"
    ",PartyPostalCode string"
    ",TargetLegalActionCode int"
    ",TargetLegalAction string"
    ",SubServicer string"
    ",HighBalanceFlag boolean"
    ",RiskDescription string"
    ",BusinessDate date"
)

##### Create - Delete Delta Table

In [0]:
partition_columns = ["BusinessDate"]
derivative_table_path = ""
 
#================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
 
 
#================== Delete From Delta Table ==================
 
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

##### Create wanted Delta Tables with selected fields & Temporary Views for each delta table

##### party

In [0]:
party = spark.sql(
f'''
SELECT *
FROM  `{env}-cdh`.primary.party
WHERE BusinessDate = {run_date!r}
'''
)

party.createOrReplaceTempView("party")



##### Create Temporary Views for Calculated fields

In [0]:
dunderlaw = spark.sql(
f'''  
    select distinct du.CDI
                    ,du.underlawtype
                    ,du.inlawflag
    from `{env}-cdh`.derivative.dunderlaw du
    where du.BusinessDate ={run_date!r}                         
'''
)
dunderlaw.createOrReplaceTempView("dunderlaw")


dktranslationmapping = spark.sql(
f'''  
    select distinct Category,FieldValue,FieldValueENG
    from `{env}-cdh`.reference.dktranslationmapping
'''
)
dktranslationmapping.createOrReplaceTempView("dktranslationmapping")

contactdetails = spark.sql(
f'''  
    SELECT distinct
    a.CDI
    ,a.SourceSystem
    ,a.PartyKey
    ,a.Country
    ,CASE   WHEN a.Country = 'CYPRUS' THEN 'Cyprus'
            ELSE g.AreaENG
    END PartyRegion,
    a.PostalCode as PartyPostalCode,
    g.AreaENG
    FROM `{env}-cdh`.primary.contactdetails a
    LEFT JOIN (
		    SELECT DISTINCT
			PostalCode
			, AreaENG
			FROM `{env}-cdh`.reference.geographicalarea
			WHERE PostalCode <> '534 38'
		    ) g
    ON substring(g.PostalCode,0,2)=substring(a.PostalCode,0,2)
    WHERE a.CommunicationFlag = 1
    and a.BusinessDate ={run_date!r}
'''
)
contactdetails.createOrReplaceTempView("contactdetails")

# p-cdh.derivative.dlegalprocessstatusdebtor

dlps = spark.sql(
f'''
SELECT cdi 
    ,TaxRegistrationNumber as LPSTaxRegistrationNumber
    , LPSCode
    , LPS
    , LPSDate
FROM (
    SELECT cdi 
        ,TaxRegistrationNumber
        , OperationalLegalStatusLevel3Code  as LPSCode
        , OperationalLegalStatusLevel3 as LPS
        , LegalProcessStatusLastChangeDate as LPSDate
        ,ROW_NUMBER() OVER (
            PARTITION BY TaxRegistrationNumber, cdi
            ORDER BY LegalProcessStatusLastChangeDate DESC NULLS LAST
        ) AS rn
    FROM `{env}-cdh`.derivative.dlegalprocessstatusdebtor
    WHERE BusinessDate = {run_date!r}
)
WHERE rn = 1
'''
)

dlps.createOrReplaceTempView("dlps")


dlps_cdi = spark.sql(
f'''
SELECT cdi, LPSCode, LPS, LPSDate
FROM (
    SELECT cdi
        , OperationalLegalStatusLevel3Code  as LPSCode
        , OperationalLegalStatusLevel3 as LPS
        , LegalProcessStatusLastChangeDate as LPSDate
        , ROW_NUMBER() OVER (
            PARTITION BY cdi
            ORDER BY LegalProcessStatusLastChangeDate DESC NULLS LAST
        ) AS rn
    FROM `{env}-cdh`.derivative.dlegalprocessstatusdebtor
    WHERE BusinessDate = {run_date!r}
)
WHERE rn = 1
'''
)
dlps_cdi.createOrReplaceTempView("dlps_cdi")





In [0]:
# High balance CDIs sourced from the external HAPS file
HighBalance = spark.sql(
f'''
SELECT DISTINCT CDI
FROM  `{env}-cdh`.externalfiles.high_balance_aggrement_haps
WHERE PartyRelationshipType = 'Primary Owner'
AND CDI IS NOT NULL
'''
)

HighBalance.createOrReplaceTempView("HighBalance")

##### party_arotron_core_wholesale

In [0]:
df_arotron_core_wholesale = spark.sql('''
select 
-- PartyId,
    party.SourceSystem
    , PortfolioKey
    , PortfolioName
    , current_timestamp as IngestionTimestamp
    , BatchId
    , party.PartyKey
    , NULL PartyKeyQCCepal
    , NULL PartyKeyProfits
    , party.PartyKey PartyKeyArotron
    , NULL PartyKeyThesis
    , NULL PartyKeyQCAlpha
    , NULL PartyKeyLMS
    , party.CDI
    , PartyTypeCode
    , PartyType
    , PartyStatusCode
    , PartyStatus
    , PartyStatusDate
    , L3869StatusCode
    , L3869Status
    , cast(L3869CourtDate as date)
    , L3869InterbankingStatusCode
    , L3869InterBankingStatus
    , case when try_cast(OCWStatus as int)=1 then true else false end as OCWFlag
    , OCWStatus
    , DeceasedFlag
    , cast(coalesce(dlps.LPSCode, dlps_cdi.LPSCode) as int)    as LPSCode
    , cast(coalesce(dlps.LPS,     dlps_cdi.LPS)     as string) as LPS
    , cast(coalesce(dlps.LPSDate, dlps_cdi.LPSDate) as date)   as LPSDate
    , Watchlist
    , FirstNameGR
    , FirstNameENG
    , LastNameGR
    , LastNameENG
    , FirstFatherNameGR
    , FirstFatherNameENG
    , LastFatherNameGR
    , LastFatherNameENG
    , FirstMotherNameGR
    , LastMotherNameGR
    , LastMotherNameENG
    , LegalNameGR
    , LegalNameENG
    , PrimaryCommercialGR
    , PrimaryCommercialENG
    , SalutationTypeCode
    , SalutationType
    , GenderCode
    , Gender
    , MaritalStatusCode
    , MaritalStatus
    , dktranslationmapping1.FieldValueENG as MaritalStatusENG
    , CitizenshipCode
    , Citizenship
    , NationalityCode
    , Nationality
    , dktranslationmapping2.FieldValueENG as NationalityENG
    , OccupationCode
    , Occupation
    , cast(BirthDate as date)
    , BirthPlace
    , CommunicationLanguageCode
    , CommunicationLanguage
    , TaxRegistrationNumber
    , TaxResidenceCountry
    , PassportNumber
    , PassportIssuingAuthority
    , PassportIssueDate
    , PassportExpiredDate
    , NationalNumber
    , NationalNumberIssuingAuthority
    , NationalNumberIssueDate
    , NationalNumberExpiredDate
    , MilitaryNumber
    , MilitaryAuthority
    , MilitaryIssueDate
    , MilitaryExpiredDate
    , SocialSecurityNumber
    , NumberOfChildren
    , ChildrenCertification
    , ChildrenCertificationIssuingAuthority
    , cast(EnterpriseStartDate as date)
    , EnterpriseEndDate
    , EconomicActivityCode
    , EconomicActivity
    , BranchActivityCode
    , BranchActivity
    , GeneralCommercialRegister
    , NumberofEmployees
    , TotalIncome
    , BalanceSheetTotalAmount
    , AnnualTurnoverAmount
    , GroupCode
    , GroupName
    , GroupCodePortfolio
    , PartyBankruptcyStage
    , CovidFlag
    , CovidCategoryCode
    , CovidCategory
    , AffectedCategoryCode
    , AffectedCategory
    , MonitoringUnitCode
    , MonitoringUnit
    , CreditRatingTypeCode
    , CreditRatingType
    , EducationLevelCode
    , EducationLevel
    , BalanceCategoryCode
    , BalanceCategory
    , ExtraSegmentCode
    , ExtraSegment
    , SegmentCode
    , Segment
    , SubSegmentCode
    , SubSegment
    , LegalFormCode
    , LegalForm
    , IsWholeshaleFlag as IsWholesaleFlag
    , EmploymentStatusCode
    , EmploymentStatus 
    , EmploymentStatusDate
    , CreatedBy
    , CreatedByDate
    , LastUpdatedBy
    , LastUpdatedByDate
    , ResubmissionStatus
    , N4605Flag
    , ConnectedBorrowerFlag
    , StaffFlag
    , L3869BorrowerHasOver50KBalance
    , case when dunderlaw.inlawflag=1 then true else false end L3869Flag
    ,contactdetails.PartyRegion
    , contactdetails.PartyPostalCode
    , TargetLegalActionCode
    , TargetLegalAction
    , SubServicer
    , NULL HighBalanceFlag
    , NULL RiskDescription
    , BusinessDate 
from party
left join dunderlaw
on party.cdi=dunderlaw.cdi and dunderlaw.underlawtype="N3869"
left join dktranslationmapping dktranslationmapping1
on dktranslationmapping1.FieldValue=party.MaritalStatus
and dktranslationmapping1.Category='MaritalStatus' 
left join dktranslationmapping dktranslationmapping2
on dktranslationmapping2.FieldValue=party.Nationality
and dktranslationmapping2.Category='Nationality'
left join contactdetails 
on contactdetails.SourceSystem=party.SourceSystem
and contactdetails.PartyKey=party.PartyKey
left join dlps
    on dlps.LPSTaxRegistrationNumber = party.TaxRegistrationNumber
   and dlps.CDI = party.CDI
left join dlps_cdi
    on dlps_cdi.CDI = party.CDI
where party.SourceSystem = 'arotron' 
and PartyType = 'Organization'
''')

df_arotron_core_wholesale.createOrReplaceTempView("df_arotron_core_wholesale")

##### party_lms

In [0]:
df_lms = spark.sql('''
select 
-- PartyId,
    party.SourceSystem
    , PortfolioKey
    , PortfolioName
    , current_timestamp as IngestionTimestamp
    , BatchId
    , party.PartyKey
    , NULL PartyKeyQCCepal
    , NULL PartyKeyProfits
    , NULL PartyKeyArotron
    , NULL PartyKeyThesis
    , NULL PartyKeyQCAlpha
    , party.PartyKey PartyKeyLMS
    , party.CDI
    , PartyTypeCode
    , PartyType
    , PartyStatusCode
    , PartyStatus
    , PartyStatusDate
    , L3869StatusCode
    , L3869Status
    , cast(L3869CourtDate as date)
    , L3869InterbankingStatusCode
    , L3869InterBankingStatus
    , case when try_cast(OCWStatus as int)=1 then true else false end as OCWFlag
    , OCWStatus
    , DeceasedFlag
    , dlps.LPSCode
    , dlps.LPS
    , dlps.LPSDate
    , Watchlist
    , FirstNameGR
    , FirstNameENG
    , LastNameGR
    , LastNameENG
    , FirstFatherNameGR
    , FirstFatherNameENG
    , LastFatherNameGR
    , LastFatherNameENG
    , FirstMotherNameGR
    , LastMotherNameGR
    , LastMotherNameENG
    , LegalNameGR
    , LegalNameENG
    , PrimaryCommercialGR
    , PrimaryCommercialENG
    , SalutationTypeCode
    , SalutationType
    , GenderCode
    , Gender
    , MaritalStatusCode
    , MaritalStatus
    , dktranslationmapping1.FieldValueENG as MaritalStatusENG
    , CitizenshipCode
    , Citizenship
    , NationalityCode
    , Nationality
    , dktranslationmapping2.FieldValueENG as NationalityENG
    , OccupationCode
    , Occupation
    , cast(BirthDate as date)
    , BirthPlace
    , CommunicationLanguageCode
    , CommunicationLanguage
    , TaxRegistrationNumber
    , TaxResidenceCountry
    , PassportNumber
    , PassportIssuingAuthority
    , PassportIssueDate
    , PassportExpiredDate
    , NationalNumber
    , NationalNumberIssuingAuthority
    , NationalNumberIssueDate
    , NationalNumberExpiredDate
    , MilitaryNumber
    , MilitaryAuthority
    , MilitaryIssueDate
    , MilitaryExpiredDate
    , SocialSecurityNumber
    , NumberOfChildren
    , ChildrenCertification
    , ChildrenCertificationIssuingAuthority
    , cast(EnterpriseStartDate as date)
    , EnterpriseEndDate
    , EconomicActivityCode
    , EconomicActivity
    , BranchActivityCode
    , BranchActivity
    , GeneralCommercialRegister
    , NumberofEmployees
    , TotalIncome
    , BalanceSheetTotalAmount
    , AnnualTurnoverAmount
    , GroupCode
    , GroupName
    , GroupCodePortfolio
    , PartyBankruptcyStage
    , CovidFlag
    , CovidCategoryCode
    , CovidCategory
    , AffectedCategoryCode
    , AffectedCategory
    , MonitoringUnitCode
    , MonitoringUnit
    , CreditRatingTypeCode
    , CreditRatingType
    , EducationLevelCode
    , EducationLevel
    , BalanceCategoryCode
    , BalanceCategory
    , ExtraSegmentCode
    , ExtraSegment
    , SegmentCode
    , Segment
    , SubSegmentCode
    , SubSegment
    , LegalFormCode
    , LegalForm
    , IsWholeshaleFlag as IsWholesaleFlag
    , EmploymentStatusCode
    , EmploymentStatus
    , EmploymentStatusDate
    , CreatedBy
    , CreatedByDate
    , LastUpdatedBy
    , LastUpdatedByDate
    , ResubmissionStatus
    , N4605Flag
    , ConnectedBorrowerFlag
    , StaffFlag
    , L3869BorrowerHasOver50KBalance
    , case when dunderlaw.inlawflag=1 then true else false end L3869Flag
    , contactdetails.PartyRegion
    , contactdetails.PartyPostalCode
    , TargetLegalActionCode
    , TargetLegalAction
    , SubServicer
    , NULL HighBalanceFlag
    , NULL RiskDescription
    , BusinessDate 
from party
left join dunderlaw
on party.cdi=dunderlaw.cdi and dunderlaw.underlawtype="N3869"
left join dktranslationmapping dktranslationmapping1
on dktranslationmapping1.FieldValue=party.MaritalStatus
and dktranslationmapping1.Category='MaritalStatus' 
left join dktranslationmapping dktranslationmapping2
on dktranslationmapping2.FieldValue=party.Nationality
and dktranslationmapping2.Category='Nationality'
left join contactdetails contactdetails
on contactdetails.SourceSystem=party.SourceSystem
and contactdetails.PartyKey=party.PartyKey
left join dlps
    on dlps.LPSTaxRegistrationNumber = party.TaxRegistrationNumber
   and dlps.CDI = party.CDI
left join dlps_cdi
    on dlps_cdi.CDI = party.CDI
where party.SourceSystem = 'lms' 
''')

df_lms.createOrReplaceTempView("df_lms")

##### party_qca_core_retail

In [0]:
df_qca_core_retail = spark.sql('''
select 
    party.SourceSystem
    , PortfolioKey
    , PortfolioName
    , current_timestamp as IngestionTimestamp
    , BatchId
    , party.PartyKey
    , NULL PartyKeyQCCepal
    , NULL PartyKeyProfits
    , NULL PartyKeyArotron
    , NULL PartyKeyThesis
    , party.PartyKey PartyKeyQCAlpha
    , NULL PartyKeyLMS
    , party.CDI
    , PartyTypeCode
    , PartyType
    , PartyStatusCode
    , PartyStatus
    , PartyStatusDate
    , L3869StatusCode
    , L3869Status
    , cast(L3869CourtDate as date)
    , L3869InterbankingStatusCode
    , L3869InterBankingStatus
    , case when try_cast(OCWStatus as int)=1 then true else false end as OCWFlag
    , OCWStatus
    , DeceasedFlag   
    , dlps.LPSCode
    , dlps.LPS
    , dlps.LPSDate
    , Watchlist
    , FirstNameGR
    , FirstNameENG
    , LastNameGR
    , LastNameENG
    , FirstFatherNameGR
    , FirstFatherNameENG
    , LastFatherNameGR
    , LastFatherNameENG
    , FirstMotherNameGR
    , LastMotherNameGR
    , LastMotherNameENG
    , LegalNameGR
    , LegalNameENG
    , PrimaryCommercialGR
    , PrimaryCommercialENG
    , SalutationTypeCode
    , SalutationType
    , GenderCode
    , Gender
    , MaritalStatusCode
    , MaritalStatus
    , dktranslationmapping1.FieldValueENG as MaritalStatusENG
    , CitizenshipCode
    , Citizenship
    , NationalityCode
    , Nationality
    , dktranslationmapping2.FieldValueENG NationalityENG
    , OccupationCode
    , Occupation
    , cast(BirthDate as date)
    , BirthPlace
    , CommunicationLanguageCode
    , CommunicationLanguage
    , TaxRegistrationNumber
    , TaxResidenceCountry
    , PassportNumber
    , PassportIssuingAuthority
    , PassportIssueDate
    , PassportExpiredDate
    , NationalNumber
    , NationalNumberIssuingAuthority
    , NationalNumberIssueDate
    , NationalNumberExpiredDate
    , MilitaryNumber
    , MilitaryAuthority
    , MilitaryIssueDate
    , MilitaryExpiredDate
    , SocialSecurityNumber
    , NumberOfChildren
    , ChildrenCertification
    , ChildrenCertificationIssuingAuthority
    , cast(EnterpriseStartDate as date)
    , EnterpriseEndDate
    , EconomicActivityCode
    , EconomicActivity
    , BranchActivityCode
    , BranchActivity
    , GeneralCommercialRegister
    , NumberofEmployees
    , TotalIncome
    , BalanceSheetTotalAmount
    , AnnualTurnoverAmount
    , GroupCode
    , GroupName
    , GroupCodePortfolio
    , PartyBankruptcyStage
    , CovidFlag
    , CovidCategoryCode
    , CovidCategory
    , AffectedCategoryCode
    , AffectedCategory
    , MonitoringUnitCode
    , MonitoringUnit
    , CreditRatingTypeCode
    , CreditRatingType
    , EducationLevelCode
    , EducationLevel
    , BalanceCategoryCode
    , BalanceCategory
    , ExtraSegmentCode
    , ExtraSegment
    , SegmentCode
    , Segment
    , SubSegmentCode
    , SubSegment
    , LegalFormCode
    , LegalForm
    , IsWholeshaleFlag as IsWholesaleFlag
    , EmploymentStatusCode
    , EmploymentStatus
    , EmploymentStatusDate
    , CreatedBy
    , CreatedByDate
    , LastUpdatedBy
    , LastUpdatedByDate
    , ResubmissionStatus
    , N4605Flag
    , ConnectedBorrowerFlag
    , StaffFlag
    , L3869BorrowerHasOver50KBalance
    , case when dunderlaw.inlawflag=1 then true else false end L3869Flag
    , contactdetails.PartyRegion
    , contactdetails.PartyPostalCode
    , TargetLegalActionCode
    , TargetLegalAction
    , SubServicer
    , case when hb.CDI is not null then true else false end HighBalanceFlag
    , NULL RiskDescription
    , BusinessDate 
from party
left join dunderlaw
on party.cdi=dunderlaw.cdi and dunderlaw.underlawtype="N3869"
left join dktranslationmapping dktranslationmapping1
on dktranslationmapping1.FieldValue=party.MaritalStatus
and dktranslationmapping1.Category='MaritalStatus' 
left join dktranslationmapping dktranslationmapping2
on dktranslationmapping2.FieldValue=party.Nationality
and dktranslationmapping2.Category='Nationality'
left join contactdetails contactdetails
on contactdetails.SourceSystem=party.SourceSystem
and contactdetails.PartyKey=party.PartyKey
left join HighBalance hb
on hb.CDI = party.CDI
left join dlps
    on dlps.LPSTaxRegistrationNumber = party.TaxRegistrationNumber
   and dlps.CDI = party.CDI
left join dlps_cdi
    on dlps_cdi.CDI = party.CDI
where party.SourceSystem = 'qca'
and PortfolioName = 'Core Retail'
''')


df_qca_core_retail.createOrReplaceTempView("df_qca_core_retail")

##### party_profits, party_qc, party_qca, party_thesis

In [0]:

party_qca = spark.sql(
'''
select * from party
where SourceSystem = 'qca'
'''
)
party_qca.createOrReplaceTempView("party_qca")

party_thesis = spark.sql(
'''
select * from party
where SourceSystem = 'thesis'
'''
)
 
party_thesis.createOrReplaceTempView("party_thesis")

party_profits = spark.sql('''
select * from party
where SourceSystem = 'profits'
qualify row_number() over (partition by PartyKey order by 1) = 1
''')

party_profits.createOrReplaceTempView("party_profits")



In [0]:
df_profits_qc_qca_thesis = spark.sql('''
select 
    party_profits.SourceSystem
    , party_profits.PortfolioKey
    , party_profits.PortfolioName
    , current_timestamp as IngestionTimestamp
    , party_profits.BatchId
    , party_profits.PartyKey
    , NULL PartyKeyQCCepal
    , party_profits.PartyKey PartyKeyProfits
    , NULL PartyKeyArotron
    , party_thesis.PartyKey PartyKeyThesis
    , party_qca.PartyKey PartyKeyQCAlpha
    , NULL PartyKeyLMS
    , party_profits.CDI
    , party_profits.PartyTypeCode
    , party_profits.PartyType
    , party_profits.PartyStatusCode
    , party_profits.PartyStatus
    , party_profits.PartyStatusDate
    , party_qca.L3869StatusCode
    , party_qca.L3869Status
    , cast(party_qca.L3869CourtDate as date)
    , party_qca.L3869InterbankingStatusCode
    , party_qca.L3869InterBankingStatus
    , case when try_cast(party_profits.OCWStatus as int)=1 then true else false end as OCWFlag
    , party_profits.OCWStatus
    , party_profits.DeceasedFlag
    , cast(coalesce(dlps.LPSCode, dlps_cdi.LPSCode) as int)    as LPSCode
    , cast(coalesce(dlps.LPS,     dlps_cdi.LPS)     as string) as LPS
    , cast(coalesce(dlps.LPSDate, dlps_cdi.LPSDate) as date)   as LPSDate
    , party_qca.Watchlist
    , party_profits.FirstNameGR
    , party_profits.FirstNameENG
    , party_profits.LastNameGR
    , party_profits.LastNameENG
    , party_profits.FirstFatherNameGR
    , party_profits.FirstFatherNameENG
    , party_profits.LastFatherNameGR
    , party_profits.LastFatherNameENG
    , party_profits.FirstMotherNameGR
    , party_profits.LastMotherNameGR
    , party_profits.LastMotherNameENG
    , party_profits.LegalNameGR
    , party_profits.LegalNameENG
    , party_profits.PrimaryCommercialGR
    , party_profits.PrimaryCommercialENG
    , party_profits.SalutationTypeCode
    , party_profits.SalutationType
    , party_profits.GenderCode
    , party_profits.Gender
    , party_profits.MaritalStatusCode
    , party_profits.MaritalStatus
    , dktranslationmapping1.FieldValueENG as MaritalStatusENG
    , party_profits.CitizenshipCode
    , party_profits.Citizenship
    , party_profits.NationalityCode
    , party_profits.Nationality
    , dktranslationmapping2.FieldValueENG as NationalityENG
    , party_profits.OccupationCode
    , party_profits.Occupation
    , cast(party_profits.BirthDate as date)
    , party_profits.BirthPlace
    , party_profits.CommunicationLanguageCode
    , party_profits.CommunicationLanguage
    , party_profits.TaxRegistrationNumber
    , party_profits.TaxResidenceCountry
    , party_qca.PassportNumber --new
    , party_qca.PassportIssuingAuthority --new
    , party_qca.PassportIssueDate -- new
    , party_qca.PassportExpiredDate --new
    , party_profits.NationalNumber
    , party_profits.NationalNumberIssuingAuthority
    , party_profits.NationalNumberIssueDate
    , party_profits.NationalNumberExpiredDate
    , party_qca.MilitaryNumber
    , party_qca.MilitaryAuthority
    , party_qca.MilitaryIssueDate
    , party_qca.MilitaryExpiredDate
    , party_qca.SocialSecurityNumber
    , party_profits.NumberOfChildren
    , party_qca.ChildrenCertification 
    , party_qca.ChildrenCertificationIssuingAuthority
    , cast(party_profits.EnterpriseStartDate as date)
    , party_profits.EnterpriseEndDate
    , party_profits.EconomicActivityCode
    , party_profits.EconomicActivity
    , party_profits.BranchActivityCode
    , party_profits.BranchActivity
    , party_profits.GeneralCommercialRegister
    , party_profits.NumberofEmployees
    , party_qca.TotalIncome
    , party_profits.BalanceSheetTotalAmount
    , party_profits.AnnualTurnoverAmount
    , party_profits.GroupCode
    , party_profits.GroupName
    , party_profits.GroupCodePortfolio
    , party_qca.PartyBankruptcyStage--new
    , NULL CovidFlag--new
    , NULL CovidCategoryCode--new
    , NULL CovidCategory--new
    , NULL AffectedCategoryCode--new
    , NULL AffectedCategory--new
    , party_profits.MonitoringUnitCode
    , party_profits.MonitoringUnit
    , party_profits.CreditRatingTypeCode
    , party_profits.CreditRatingType
    , party_profits.EducationLevelCode
    , party_profits.EducationLevel
    , party_qca.BalanceCategoryCode --new
    , party_qca.BalanceCategory--new
    , party_qca.ExtraSegmentCode--new
    , party_qca.ExtraSegment--new
    , party_qca.SegmentCode--new
    , party_qca.Segment--new
    , party_qca.SubSegmentCode--new
    , party_qca.SubSegment--new
    , party_profits.LegalFormCode
    , party_profits.LegalForm
    , party_qca.IsWholeshaleFlag as IsWholesaleFlag
    , party_profits.EmploymentStatusCode
    , party_profits.EmploymentStatus
    , party_profits.EmploymentStatusDate
    , party_profits.CreatedBy
    , party_profits.CreatedByDate
    , party_profits.LastUpdatedBy
    , party_profits.LastUpdatedByDate
    , party_qca.ResubmissionStatus
    , party_qca.N4605Flag
    , party_qca.ConnectedBorrowerFlag
    , party_qca.StaffFlag 
    , party_qca.L3869BorrowerHasOver50KBalance
    , case when dunderlaw.inlawflag=1 then true else false end L3869Flag
    , contactdetails.PartyRegion
    , contactdetails.PartyPostalCode
    , party_qca.TargetLegalActionCode TargetLegalActionCode  -- new
    , party_qca.TargetLegalAction  TargetLegalAction  -- new
    , party_qca.SubServicer as SubServicer  -- new
    , case when hb.CDI is not null then true else false end HighBalanceFlag
    , party_profits.RiskDescription
    , party_profits.BusinessDate 
from party_profits 
left join party_qca
on party_profits.cdi = party_qca.cdi
left join party_thesis
on party_profits.cdi = party_thesis.cdi  
left join dunderlaw
on party_profits.cdi=dunderlaw.cdi and dunderlaw.UnderLawType="N3869"
left join dktranslationmapping dktranslationmapping1
on dktranslationmapping1.FieldValue=party_profits.MaritalStatus
and dktranslationmapping1.Category='MaritalStatus' 
left join dktranslationmapping dktranslationmapping2
on dktranslationmapping2.FieldValue=party_profits.Nationality
and dktranslationmapping2.Category='Nationality'
left join contactdetails contactdetails
on contactdetails.SourceSystem=party_profits.SourceSystem
and contactdetails.PartyKey=party_profits.PartyKey
left join HighBalance hb
on hb.CDI = party_profits.CDI
left join dlps
    on dlps.LPSTaxRegistrationNumber = party_profits.TaxRegistrationNumber
   and dlps.CDI = party_profits.CDI
left join dlps_cdi
    on dlps_cdi.CDI = party_profits.CDI
''')

df_profits_qc_qca_thesis.createOrReplaceTempView("df_profits_qc_qca_thesis")

##### party_thesis_only

In [0]:
party_thesis_only  = spark.sql(
'''
select party_thesis.* 
from party_thesis
left join party_profits
on party_thesis.cdi = party_profits.cdi
where party_profits.cdi is null
'''
)

party_thesis_only.createOrReplaceTempView("party_thesis_only")

In [0]:
df_thesis = spark.sql('''
select 
     party_thesis.SourceSystem
    , PortfolioKey
    , PortfolioName
    , current_timestamp as IngestionTimestamp 
    , BatchId
    , party_thesis.PartyKey
    , NULL PartyKeyQCCepal
    , NULL PartyKeyProfits
    , NULL PartyKeyArotron
    , party_thesis.PartyKey PartyKeyThesis
    , NULL PartyKeyQCAlpha
    , NULL PartyKeyLMS
    , party_thesis.CDI
    , PartyTypeCode
    , PartyType
    , PartyStatusCode
    , PartyStatus
    , PartyStatusDate
    , L3869StatusCode
    , L3869Status
    , cast(L3869CourtDate as date)
    , L3869InterbankingStatusCode
    , L3869InterBankingStatus
    , case when try_cast(OCWStatus as int)=1 then true else false end as OCWFlag
    , OCWStatus
    , DeceasedFlag
    , dlps.LPSCode
    , dlps.LPS
    , dlps.LPSDate
    , Watchlist
    , FirstNameGR
    , FirstNameENG
    , LastNameGR
    , LastNameENG
    , FirstFatherNameGR
    , FirstFatherNameENG
    , LastFatherNameGR
    , LastFatherNameENG
    , FirstMotherNameGR
    , LastMotherNameGR
    , LastMotherNameENG
    , LegalNameGR
    , LegalNameENG
    , PrimaryCommercialGR
    , PrimaryCommercialENG
    , SalutationTypeCode
    , SalutationType
    , GenderCode
    , Gender
    , MaritalStatusCode
    , MaritalStatus
    , dktranslationmapping1.FieldValueENG as MaritalStatusENG
    , CitizenshipCode
    , Citizenship
    , NationalityCode
    , Nationality
    , dktranslationmapping2.FieldValueENG as NationalityENG
    , OccupationCode
    , Occupation
    , cast(BirthDate as date)
    , BirthPlace
    , CommunicationLanguageCode
    , CommunicationLanguage
    , TaxRegistrationNumber
    , TaxResidenceCountry
    , PassportNumber
    , PassportIssuingAuthority
    , PassportIssueDate
    , PassportExpiredDate
    , NationalNumber
    , NationalNumberIssuingAuthority
    , NationalNumberIssueDate
    , NationalNumberExpiredDate
    , MilitaryNumber
    , MilitaryAuthority
    , MilitaryIssueDate
    , MilitaryExpiredDate
    , SocialSecurityNumber
    , NumberOfChildren
    , ChildrenCertification
    , ChildrenCertificationIssuingAuthority
    , cast(EnterpriseStartDate as date)
    , EnterpriseEndDate
    , EconomicActivityCode
    , EconomicActivity
    , BranchActivityCode
    , BranchActivity
    , GeneralCommercialRegister
    , NumberofEmployees
    , TotalIncome
    , BalanceSheetTotalAmount
    , AnnualTurnoverAmount
    , GroupCode
    , GroupName
    , GroupCodePortfolio
    , PartyBankruptcyStage
    , CovidFlag
    , CovidCategoryCode
    , CovidCategory
    , AffectedCategoryCode
    , AffectedCategory
    , MonitoringUnitCode
    , MonitoringUnit
    , CreditRatingTypeCode
    , CreditRatingType
    , EducationLevelCode
    , EducationLevel
    , BalanceCategoryCode
    , BalanceCategory
    , ExtraSegmentCode
    , ExtraSegment
    , SegmentCode
    , Segment
    , SubSegmentCode
    , SubSegment
    , LegalFormCode
    , LegalForm
    , IsWholeshaleFlag as IsWholesaleFlag
    , EmploymentStatusCode
    , EmploymentStatus
    , EmploymentStatusDate
    , CreatedBy
    , CreatedByDate
    , LastUpdatedBy
    , LastUpdatedByDate
    , ResubmissionStatus
    , N4605Flag
    , ConnectedBorrowerFlag
    , StaffFlag
    , L3869BorrowerHasOver50KBalance
    , case when dunderlaw.inlawflag=1 then true else false end L3869Flag
    , contactdetails.PartyRegion
    , contactdetails.PartyPostalCode
    , TargetLegalActionCode
    , TargetLegalAction
    , SubServicer
    , case when hb.CDI is not null then true else false end HighBalanceFlag
    , NULL RiskDescription
    , BusinessDate 
from party_thesis_only as party_thesis  
left join dunderlaw
on party_thesis.cdi=dunderlaw.cdi and dunderlaw.underlawtype="N3869"
left join dktranslationmapping dktranslationmapping1
on dktranslationmapping1.FieldValue=party_thesis.MaritalStatus
and dktranslationmapping1.Category='MaritalStatus' 
left join dktranslationmapping dktranslationmapping2
on dktranslationmapping2.FieldValue=party_thesis.Nationality
and dktranslationmapping2.Category='Nationality'
left join contactdetails contactdetails
on contactdetails.SourceSystem=party_thesis.SourceSystem
and contactdetails.PartyKey=party_thesis.PartyKey
left join HighBalance hb
on hb.CDI = party_thesis.CDI
left join dlps
    on dlps.LPSTaxRegistrationNumber = party_thesis.TaxRegistrationNumber
   and dlps.CDI = party_thesis.CDI
left join dlps_cdi
    on dlps_cdi.CDI = party_thesis.CDI
''')

df_thesis.createOrReplaceTempView("df_thesis")

##### Define some subqueries from the final query as temporary views for better readability and maintenance

##### Final query with resulted DataFrame

In [0]:
final_df = spark.sql('''
    SELECT distinct *
    FROM df_arotron_core_wholesale
    UNION
    SELECT distinct *
    FROM df_lms
    UNION
    SELECT distinct *
    FROM df_qca_core_retail
    UNION
    SELECT distinct *
    FROM df_profits_qc_qca_thesis
    UNION
    SELECT distinct *
    FROM df_thesis
''')

final_df_PartyDerivativeId_primary_key_fields = [
    final_df.PartyKey,
    final_df.SourceSystem
]

final_df = (final_df.withColumn(
    "PartyDerivativeId",
    sha2(
        concat_ws(";", *final_df_PartyDerivativeId_primary_key_fields),
        256,
    )
))


In [0]:
from pyspark.sql import functions as F

columns = final_df.columns

# Define the new schema with the new column as the first column
new_columns = ["PartyDerivativeId"] + [col for col in columns if col != "PartyDerivativeId"]

# Reorder the DataFrame columns
final_df = final_df.select(new_columns)

# Align LPS column types to the target Delta schema (LPSCode int, LPSDate date)
final_df = (final_df
    .withColumn("LPSCode", F.col("LPSCode").cast("int"))
    .withColumn("LPSDate", F.col("LPSDate").cast("date"))
)

In [0]:
final_df.createOrReplaceTempView("final_df")

##### Write DataFrame to Delta Lake based on 'unity' value

In [0]:
if unity == "True":
    final_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else:
    final_df.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake